In [62]:
import pandas as pd

df = pd.read_csv("../data/contract_evaluation_dataset_sathsreehari-k.csv")
df.head()

,contract_id,contract_text,expected_apr,expected_term,expected_monthly_payment,expected_penalty
0,1,This contract is entered between Prime Auto Fi...,9.75,36.0,825.0,late payment fee of $50
1,2,This leasing agreement between ABC Motors and ...,6.20,48.0,640.0,NaN
2,3,This agreement is signed between DriveEasy Lea...,NaN,24.0,510.0,early termination fee of $300
3,4,This auto loan contract between National Motor...,11.40,60.0,940.0,NaN
4,5,The contract between CarHub Leasing and Priya ...,NaN,36.0,720.0,late payment fee of $25


In [63]:
expected_output = {
    "apr": None,
    "term_months": None,
    "monthly_payment": None,
    "penalty_clause": None
}

In [64]:
PROMPT_TEMPLATE = """
You are an information extraction system.

Extract ONLY the following fields from the contract text below:
- APR
- Term (in months)
- Monthly payment
- Penalty clause

Rules:
- Return output in valid JSON format only
- If a value is NOT explicitly mentioned, return null
- Do NOT infer, calculate, or assume any values
- Do NOT add extra fields
- Do NOT explain anything

Return JSON in this exact structure:
{{
  "apr": null,
  "term_months": null,
  "monthly_payment": null,
  "penalty_clause": null
}}
"""


In [65]:
def llm(prompt_text):
    """
    Dummy LLM response for baseline testing.
    """
    return json.dumps({
        "apr": None,
        "term_months": None,
        "monthly_payment": None,
        "penalty_clause": None
    })

In [66]:

sample_df = df.sample(5, random_state=42)
sample_df

,contract_id,contract_text,expected_apr,expected_term,expected_monthly_payment,expected_penalty
27,28,This auto lease agreement between CarEase Ltd ...,NaN,30.0,575.0,NaN
15,16,This auto financing agreement between National...,NaN,72.0,890.0,NaN
23,24,This agreement between DriveSmart Leasing and ...,NaN,12.0,390.0,NaN
17,18,This vehicle loan document between SpeedTrack ...,NaN,48.0,NaN,foreclosure penalty of $400
8,9,This lease contract signed by Urban Auto and K...,NaN,18.0,495.0,early termination fee of $200


In [67]:
def extract_fields(contract_text):
    prompt = PROMPT_TEMPLATE.format(contract_text=contract_text)
    response = llm(prompt)
    return json.loads(response)

In [68]:
import json
results = []

for _, row in sample_df.iterrows():
    extracted = extract_contract_fields(
        row["contract_text"],
        llm
    )

    results.append({
        "contract_text": row["contract_text"],
        "llm_apr": extracted["apr"],
        "llm_term": extracted["term_months"],
        "llm_payment": extracted["monthly_payment"],
        "llm_penalty": extracted["penalty_clause"],
        "expected_apr": row["expected_apr"],
        "expected_term": row["expected_term"],
        "expected_payment": row["expected_monthly_payment"],
        "expected_penalty": row["expected_penalty"]
    })

results_df = pd.DataFrame(results)
results_df

,contract_text,llm_apr,llm_term,llm_payment,llm_penalty,expected_apr,expected_term,expected_payment,expected_penalty
0,This auto lease agreement between CarEase Ltd ...,None,None,None,None,NaN,30.0,575.0,NaN
1,This auto financing agreement between National...,None,None,None,None,NaN,72.0,890.0,NaN
2,This agreement between DriveSmart Leasing and ...,None,None,None,None,NaN,12.0,390.0,NaN
3,This vehicle loan document between SpeedTrack ...,None,None,None,None,NaN,48.0,NaN,foreclosure penalty of $400
4,This lease contract signed by Urban Auto and K...,None,None,None,None,NaN,18.0,495.0,early termination fee of $200


In [69]:

sample = df.sample(5, random_state=1)
sample

,contract_id,contract_text,expected_apr,expected_term,expected_monthly_payment,expected_penalty
17,18,This vehicle loan document between SpeedTrack ...,NaN,48.0,NaN,foreclosure penalty of $400
21,22,This contract entered by CityWheels and Isha K...,NaN,NaN,640.0,NaN
10,11,This agreement between CityCar Finance and Sne...,NaN,12.0,NaN,NaN
19,20,This leasing agreement between AutoFlex Ltd an...,NaN,24.0,455.0,early exit fee of $250
14,15,This contract between Swift Motors and Riya Ba...,NaN,NaN,NaN,late payment penalty of $60


In [70]:
extracted_rows = []

for _, row in sample.iterrows():
    extracted = extract_fields(row["contract_text"])

    extracted_rows.append({
        "contract_text": row["contract_text"],

        "llm_apr": extracted["apr"],
        "llm_term": extracted["term_months"],
        "llm_payment": extracted["monthly_payment"],
        "llm_penalty": extracted["penalty_clause"],

        "expected_apr": row["expected_apr"],
        "expected_term": row["expected_term"],
        "expected_penalty": row["expected_penalty"]
    })

comparison_df = pd.DataFrame(extracted_rows)
comparison_df

,contract_text,llm_apr,llm_term,llm_payment,llm_penalty,expected_apr,expected_term,expected_penalty
0,This vehicle loan document between SpeedTrack ...,None,None,None,None,NaN,48.0,foreclosure penalty of $400
1,This contract entered by CityWheels and Isha K...,None,None,None,None,NaN,NaN,NaN
2,This agreement between CityCar Finance and Sne...,None,None,None,None,NaN,12.0,NaN
3,This leasing agreement between AutoFlex Ltd an...,None,None,None,None,NaN,24.0,early exit fee of $250
4,This contract between Swift Motors and Riya Ba...,None,None,None,None,NaN,NaN,late payment penalty of $60


***SLA EXTRACTION***

In [71]:

df=pd.read_csv('../notebook/milestone1_sathsreehari_k.csv')
df.head()

,contract_id,raw_text,apr,term_months,monthly_payment,penalty_clause,recommended_action,risk_flag,expected_apr,expected_term,expected_payment,expected_penalty,apr_score,term_score,payment_score,penalty_score,total_score,quality_percent
0,1,This contract between Prime Auto and Emily Sto...,10.49,24,959,NaN,Approve,Low,10.49,24,959,NaN,1,1,1,0,3,75.0
1,2,This contract between ABC Motors and Priya Sha...,3.78,24,804,Early termination fee $300,Approve,Low,3.78,24,804,Early termination fee $300,1,1,1,1,4,100.0
2,3,This contract between DriveEasy Finance and Sa...,5.41,24,774,Late fee $50,Reject,High,5.41,24,774,Late fee $50,1,1,1,1,4,100.0
3,4,This contract between National Motors and Alex...,5.26,48,1028,Late fee $25,Approve,High,5.26,48,1028,Late fee $25,1,1,1,1,4,100.0
4,5,This contract between National Motors and John...,5.97,36,1180,NaN,Approve,Low,5.97,36,1180,NaN,1,1,1,0,3,75.0


In [72]:
df=df.rename(columns={"penalty":"penalty_clause"})

In [73]:
df["expected_apr"]=df["apr"]
df["expected_term"]=df["term_months"]
df["expected_payment"]=df["monthly_payment"]
df["expected_penalty"]=df["penalty_clause"]

In [74]:

df[["penalty_clause","expected_penalty"]].head()

,penalty_clause,expected_penalty
0,NaN,NaN
1,Early termination fee $300,Early termination fee $300
2,Late fee $50,Late fee $50
3,Late fee $25,Late fee $25
4,NaN,NaN


In [75]:
df["apr_score"]=(df["apr"]==df["expected_apr"]).astype(int)
df["term_score"]=(df["term_months"]==df["expected_term"]).astype(int)
df["payment_score"]=(df["monthly_payment"]==df["expected_payment"]).astype(int)
df["penalty_score"]=(df["penalty_clause"]==df["expected_penalty"]).astype(int)

df["total_score"]=(df["apr_score"]+df["term_score"]+df["payment_score"]+df["penalty_score"])

In [76]:

df["quality_percent"]=(df["total_score"]/4)*100
df[["apr_score","term_score","payment_score","penalty_score","total_score","quality_percent"]].head()

,apr_score,term_score,payment_score,penalty_score,total_score,quality_percent
0,1,1,1,0,3,75.0
1,1,1,1,1,4,100.0
2,1,1,1,1,4,100.0
3,1,1,1,1,4,100.0
4,1,1,1,0,3,75.0


In [77]:

df.to_csv("milestone1_sathsreehari_k.csv",index=False)